# Data Sampling and Stratification Strategy: Asymmetric Robust Sampling Framework (ARSF)

This notebook establishes the foundational datasets for the Adaptive Extractor pipeline, specifically tailored for the **Benzimidazole Antibiotics** dataset. To ensure statistical rigor and maximize the efficiency of In-Context Learning (ICL), we employ an asymmetric sampling strategy:

### Phase 1: Mathematical Context Isolation
* **Grouping Key:** The unique identifier for a parent source is the `pdf` column (e.g., `cjoc.202200326`).
* **Atomicity:** Each scientific article (`pdf`) is an indivisible unit (atom) of sampling. All molecules, SMILES, and measurement results extracted from one article strictly fall into a single split (Train, Val, or Test) to prevent data leakage.

### Phase 2: Designing a Unified Meta-Feature Space
Each `pdf` source is projected into a feature space based on the aggregation of its rows across three categories:
* **Volume and Density:** Total number of tested compounds and chemical density.
* **Domain Heterogeneity:** Diversity and conflicting contexts (e.g., switching measurement units in `target_units`, variance of values, and distinct strains in `bacteria`).
* **Structural Complexity:** Presence of toxic numerical formats (e.g., `,` or `±`), inequalities (`>`, `<`), and the topological length of `smiles` strings.

### Phase 3 & 4: Asymmetric Split and Topological Stratification
* **Hard-Example Mining (Train Set):** Extracting the most information-dense edge cases (highest Integral Complexity Score) to serve as demonstrations for prompt optimization.
* **Stratified Random Sampling (Val/Test Sets):** Projecting remaining document meta-features into a lower-dimensional space (UMAP) to identify structural clusters, ensuring unbiased and representative evaluation datasets.

## 1. Imports and Configuration

In [1]:
import numpy as np
import pandas as pd
import re
import json
from typing import Dict, Any, List

import umap
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

In [2]:
# File paths
INPUT_CSV = "../../data/ground_truth/benzimidazole_antibiotics.csv"
OUTPUT_SPLITS = "../../data/splits/benzimidazole_pilot.json"

## 2. Data Loading and Preprocessing
We begin by loading the ground truth dataset. Preprocessing involves standardizing text fields and handling missing values to prepare for atomic group-level aggregation.

In [3]:
# Load raw data
df = pd.read_csv(INPUT_CSV)

# Preprocessing: Clean strings and handle NaNs
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
df = df.replace([r'^\s*$', 'nan', 'NaN', 'None', 'none'], pd.NA, regex=True)

print(f"Total experiments loaded: {len(df)}")
print(f"Total unique articles (PDFs): {df['pdf'].nunique()}")

## 3. Feature Engineering and Information Density Scoring
To computationally evaluate the "difficulty" of a document, we map domain-specific challenges into an n-dimensional space.
We extract raw features representing Volume, Domain Heterogeneity, and Structural Complexity. Afterward, continuous variables are scaled to a [0, 1] range using `MinMaxScaler`, and an aggregate **Integral Complexity Score** is calculated using expert-defined weights.

In [4]:
def parse_value(v: Any) -> float:
    """Extracts the first valid numeric float from a string containing noise like commas or +/-."""
    if pd.isna(v):
        return np.nan
    v_str = str(v).replace(',', '.')
    match = re.search(r'[-+]?\d*\.?\d+', v_str)
    return float(match.group()) if match else np.nan

def has_toxic_format(group: pd.DataFrame) -> int:
    """Detects presence of toxic numerical formats (e.g., ± or comma usage)."""
    toxic_mask = group['target_value'].astype(str).str.contains(r'[±,]', na=False)
    return 1 if toxic_mask.any() else 0

def calculate_raw_features(pdf: str, group: pd.DataFrame) -> Dict[str, Any]:
    # 1. Volume and Density
    n_rows = len(group)
    chem_density = group['compound_id'].nunique() / n_rows if n_rows > 0 else 0.0
    
    # 2. Domain Heterogeneity
    unit_mix = group['target_units'].nunique()
    bacteria_set = set(group['bacteria'].dropna()) | set(group['bacteria_unified'].dropna())
    bio_het = len(bacteria_set)
    
    parsed_vals = group['target_value'].apply(parse_value).dropna()
    val_var = parsed_vals.var() if len(parsed_vals) > 1 else 0.0
    if pd.isna(val_var): val_var = 0.0
    
    # 3. Structural Complexity
    toxic_formats = has_toxic_format(group)
    
    clean_relations = group['target_relation'].astype(str).str.replace("'", "").str.strip()
    ineq_ratio = (clean_relations != '=').mean()
    
    max_smiles_len = group['smiles'].astype(str).str.len().max() if not group['smiles'].isna().all() else 0
    max_bact_len = group['bacteria'].astype(str).str.len().max() if not group['bacteria'].isna().all() else 0

    return {
        'PDF': pdf,
        'N_Rows': n_rows,
        'Chem_Density': chem_density,
        'Unit_Mix': unit_mix,
        'Bio_Het': bio_het,
        'Val_Var': val_var,
        'Toxic_Formats': toxic_formats,
        'Ineq_Ratio': ineq_ratio,
        'Max_SMILES_Len': max_smiles_len,
        'Max_Bact_Len': max_bact_len
    }

In [5]:
# Extract raw features for all PDFs
candidates = []
for pdf, group in df.groupby('pdf'):
    candidates.append(calculate_raw_features(pdf, group))

results = pd.DataFrame(candidates)

# Normalize Continuous Features (MinMaxScaler)
scaler = MinMaxScaler()
scale_cols = ['N_Rows', 'Chem_Density', 'Unit_Mix', 'Bio_Het', 'Val_Var', 'Ineq_Ratio', 'Max_SMILES_Len', 'Max_Bact_Len']
results[scale_cols] = scaler.fit_transform(results[scale_cols])

# Calculate Integral Complexity Score based on feature weights
results['Score'] = (
    results['Unit_Mix'] * 5.0 +        # Highest risk: unit switching
    results['Toxic_Formats'] * 4.0 +   # High risk: parsing failures
    results['Ineq_Ratio'] * 4.0 +      # High risk: inequality logic
    results['Bio_Het'] * 3.0 +         # Medium risk: multiple strains
    results['Val_Var'] * 2.0 +         # Medium risk: large value variance
    results['Max_SMILES_Len'] * 2.0 +  # Medium risk: context overflow
    results['Max_Bact_Len'] * 1.5 +    # Low-Medium risk: noisy text
    results['N_Rows'] * 1.0 +          # Context density
    results['Chem_Density'] * 1.0      # Architectural density
)

print(f"Feature extraction and scoring complete. Total articles: {len(results)}")
display(results.sort_values('Score', ascending=False).head())

## 4. Dimensionality Reduction (UMAP) and Clustering (K-Means)
To build representative Validation and Test sets, we stratify the population based on structural and semantic similarity.
1. **Topological Projection:** We apply **UMAP** using the `euclidean` metric (as features are now continuous `[0,1]`) to project the meta-feature space into 2D.
2. **Clustering:** We apply **K-Means** ($k=4$) to partition the documents into macro-clusters.

In [6]:
# Prepare data for clustering (all scaled numeric features + toxic flag)
feature_cols = scale_cols + ['Toxic_Formats']
X = results[feature_cols].copy()

# 1. Dimensionality Reduction with UMAP
reducer = umap.UMAP(
    n_neighbors=min(15, len(X) - 1),
    min_dist=0.1,
    n_components=2,
    random_state=42,
    metric='euclidean',
    n_jobs=1
)
embedding = reducer.fit_transform(X)
results['umap_1'] = embedding[:, 0]
results['umap_2'] = embedding[:, 1]

# 2. Clustering with K-Means
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
results['Cluster'] = kmeans.fit_predict(embedding)

print(f"Clustering complete. Distribution:\n{results['Cluster'].value_counts().sort_index()}")

In [7]:
# Visualize Clusters
plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    results['umap_1'], 
    results['umap_2'], 
    c=results['Cluster'], 
    cmap='viridis', 
    alpha=0.8, 
    s=80
)
plt.title('Document Clusters (UMAP + K-Means)')
plt.xlabel('UMAP Component 1')
plt.ylabel('UMAP Component 2')
plt.colorbar(scatter, label='Cluster ID')
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

## 5. Sampling Strategy

### 5.1. Hard-Example Mining (Step-Decay Analysis)
The training pool must consist of the most information-dense edge cases to demonstrate the full extraction schema. We rank articles by their `Score` and dynamically set a threshold (e.g., 85th percentile) to isolate the "ultra-hard" examples.

In [8]:
pool = results.copy()

# Sort by Score (descending) to prioritize complex examples for training
pool = pool.sort_values('Score', ascending=False)

# Calculate dynamic threshold (Top 15% most complex articles)
THRESHOLD = np.percentile(pool['Score'], 85)

plt.figure(figsize=(10, 4))
plt.plot(pool['Score'].values, 'o-', linewidth=2, markersize=5)
plt.xlabel('Sample Index', fontsize=12)
plt.ylabel('Complexity Score', fontsize=12)
plt.title('Score Decay - Threshold Detection for N_demo', fontsize=14)

plt.axhline(y=THRESHOLD, color='r', linestyle='--', alpha=0.7, label=f'Threshold (Score >= {THRESHOLD:.2f})')

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.2. Train Set Extraction
Based on the dynamic threshold, we isolate the `train_auto` pool. The single most complex article (Top-1) is reserved for the `train_manual` baseline.

In [9]:
# 1. Training Set (Hard-Example Mining)
train_pool = pool[pool['Score'] >= THRESHOLD]

train_auto_pdfs = train_pool['PDF'].tolist()
train_manual_pdfs = train_pool.head(1)['PDF'].tolist() # Top-1 subset for 1-shot manual

print(f"N_demo (Auto Train Pool) size: {len(train_auto_pdfs)}")
print(f"Manual Baseline size:          {len(train_manual_pdfs)}")

# Remove ALL training samples from the remaining pool to avoid data leakage
remaining_pool = pool[~pool['PDF'].isin(train_auto_pdfs)]

### 5.3. Validation and Test Sets (Stratified by Cluster)
We sample proportions from the remaining pool for the Validation and Test sets using Stratified Random Sampling to reflect the natural structural distribution.

In [10]:
# 2. Validation and Test Sets
cluster_counts = remaining_pool['Cluster'].value_counts(normalize=True)

# Allocate ~20% to Val and ~20% to Test from the overall pool dynamically
sample_size = max(3, int(len(pool) * 0.20))

cluster_plan = (cluster_counts * sample_size).round().astype(int)
diff = sample_size - cluster_plan.sum()
if diff != 0:
    idx = cluster_plan.idxmax()
    cluster_plan[idx] += diff

val_pdfs = []
test_pdfs = []

temp_pool = remaining_pool.copy()

# Create Validation Set
for cluster_id, count in cluster_plan.items():
    cluster_data = temp_pool[temp_pool['Cluster'] == cluster_id]
    selected = cluster_data.sample(n=min(count, len(cluster_data)), random_state=42)
    val_pdfs.extend(selected['PDF'].tolist())
    temp_pool = temp_pool[~temp_pool['PDF'].isin(selected['PDF'])]

# Create Test Set
for cluster_id, count in cluster_plan.items():
    cluster_data = temp_pool[temp_pool['Cluster'] == cluster_id]
    selected = cluster_data.sample(n=min(count, len(cluster_data)), random_state=42)
    test_pdfs.extend(selected['PDF'].tolist())
    temp_pool = temp_pool[~temp_pool['PDF'].isin(selected['PDF'])]

print(f"Sampling complete:")
print(f"  Validation:     {len(val_pdfs)}")
print(f"  Test:           {len(test_pdfs)}")

## 6. Dataset Serialization
We serialize the finalized dataset splits into a JSON artifact.

In [11]:
splits = {
    "train_manual": train_manual_pdfs,
    "train": train_auto_pdfs,
    "val": val_pdfs,
    "test": test_pdfs
}

with open(OUTPUT_SPLITS, 'w', encoding='utf-8') as f:
    json.dump(splits, f, indent=4, ensure_ascii=False)

print(f"Splits saved to {OUTPUT_SPLITS}")

## 7. Integrity and Sanity Checks
Critical integrity checks to prevent data leakage and ensure statistical validity:
1. **Isolation Audit:** Ensure zero overlap between Train, Val, and Test PDF identifiers.
2. **Representativeness Audit:** Compare cluster proportions of Val and Test sets against the General Population.

In [12]:
# 1. Isolation Audit (Data Leakage Check)
train_set = set(train_auto_pdfs)
val_set = set(val_pdfs)
test_set = set(test_pdfs)

assert len(train_set.intersection(val_set)) == 0, "Leakage between Train and Val!"
assert len(train_set.intersection(test_set)) == 0, "Leakage between Train and Test!"
assert len(val_set.intersection(test_set)) == 0, "Leakage between Val and Test!"

# 2. Distribution Audit
pop_dist = pool['Cluster'].value_counts(normalize=True).sort_index()

print("--- Cluster Distribution Analysis ---")
print(f"General Population (N={len(pool)}):\n{pop_dist}\n")

def check_distribution(pdfs, name):
    if not pdfs: return
    subset = results[results['PDF'].isin(pdfs)]
    dist = subset['Cluster'].value_counts(normalize=True).sort_index()
    print(f"{name} Set (N={len(pdfs)}):\n{dist}\n")

check_distribution(val_pdfs, "Validation")
check_distribution(test_pdfs, "Test")

print("--- Integrity Checks Passed ---")
print("✓ Success: Zero data leakage (strict mathematical isolation by PDF).")
print("✓ Success: Validation and Test sets reflect population proportions.")

## 8. Reproducible Integrity Checks (from Serialized Splits)
Demonstrating that sampling results are completely reproducible independent of in-memory execution state.

In [13]:
# Load splits from JSON
with open(OUTPUT_SPLITS, 'r', encoding='utf-8') as f:
    loaded_splits = json.load(f)

print(f"Loaded splits from {OUTPUT_SPLITS}")
print(f"  train_manual: {len(loaded_splits['train_manual'])} articles")
print(f"  train_auto:   {len(loaded_splits['train'])} articles")
print(f"  val:          {len(loaded_splits['val'])} articles")
print(f"  test:         {len(loaded_splits['test'])} articles")

# Verify subset condition
assert set(loaded_splits['train_manual']).issubset(set(loaded_splits['train'])), "Manual train must be a subset of Auto train"
print("✓ Success: Results are verified and reproducible from serialized splits file.")